# First model: Basic Gaussian Mixture

While the end-goal of this project is to have a functioning Hidden Markov Model to capture the temporal behaviour of our crops, it is first necessary for us to establish a baseline model operating under the assumption of statistical independence. The inherent problem of this project is unsupervised, therefore we consider a range of unsupervised models, starting off with clustering. The approach of fitting a GMM in this notebook answers the question: **Are conditions behaving abnormally right now?**

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture 
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from data import utils

## Load Data

In [3]:
non_spectral_df = pd.read_csv("../data/raw/non_spectral_df.csv", index_col=0)
non_spectral_df["Timestamp"] = pd.to_datetime(non_spectral_df["Timestamp"])

spectral_df = pd.read_csv("../data/raw/spectral_df.csv", index_col=0)
spectral_df["Timestamp"] = pd.to_datetime(spectral_df["Timestamp"])

grouped_spectral_df = pd.read_csv("../data/raw/grouped_spectral_df.csv", index_col=0)
grouped_spectral_df["Timestamp"] = pd.to_datetime(grouped_spectral_df["Timestamp"])



In [4]:
# Engineered features
eng_non_spectral_df  = pd.read_csv("../data/processed/eng_non_spectral_df.csv", index_col=0)
eng_non_spectral_df["Timestamp"] = pd.to_datetime(eng_non_spectral_df["Timestamp"])
eng_non_spectral_df["in_season"] = eng_non_spectral_df["in_season"].astype('category')
eng_non_spectral_df["Growth_Stage"] = eng_non_spectral_df["Growth_Stage"].astype('category')


In [7]:
# Rolling statistics and lagged variables
roll_lag_df = pd.read_csv("../data/processed/roll_df.csv", index_col=0)
roll_lag_df["Timestamp"] = pd.to_datetime(roll_lag_df["Timestamp"])

In [8]:
non_spectral_df = non_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)

grouped_spectral_df = grouped_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)



### Extracting Anomaly Data

An immediate answer to the question "Why fit a GMM?" is apparent here. Our anomly years found in the EDA process are _not_ a sequential set (2018-2019, 2023, maybe 2024). This means there will be a time gap between the training years. In a **sequential** model like an HMM, this would be a problem, but a GMM makes the assumption of statistical independence, so the gap doesn't matter. 

In [9]:
# Extract extreme drought years 2018, 2019

def get_anomaly_df(df):
    # df = rem_year(df, 2025)
    mask = (df["Ag_year"].isin([2018, 2019, 2023, 2024]))
    return df[~mask].copy(), df[mask].copy()

def rem_year(df, year, *years):
    all_years = [year, *years]
    mask = df["Timestamp"].dt.year.isin(all_years)
    return df[~mask]

In [10]:
norm_ns_df, anom_ns_df = get_anomaly_df(non_spectral_df)
norm_s_df, anom_s_df = get_anomaly_df(grouped_spectral_df[["Timestamp", "Ag_year", "NDVI_mean", "NDWI_mean", "NDRE_mean"]])

norm_eng_ns_df, anom_eng_ns_df = get_anomaly_df(eng_non_spectral_df)
norm_roll_lag_df, anom_roll_lag_df = get_anomaly_df(roll_lag_df)


KeyError: 'Ag_year'

### Interpolation

We present the first issue of our data using a simple merge between the non-spectral and spectral datasets.

In [ ]:
master_raw_df = grouped_spectral_df.merge(non_spectral_df, on="Timestamp" ,how='right')
master_raw_df.head(7)

You will quickly notice that our spectral indices contain lots of `NaN` observations. This is due to the almost-weekly passes of the Sentinel-2 mission, as well as cloud-masking applied in our pre-processing steps. Luckily, NDVI follows a pretty general pattern and is not prone to extreme oscilations from one moment to the next. It is therefore not a bad idea to use the built-in **Cubic Hermit Spline** wrapper to `scipy` in `pandas` to account for these unobserved images. 

Note we also use **backfilling** to account for the missing observations at the start of our datasets, as the spectral indices are unlikely to vary much in such a short period.

In [ ]:
def get_interpolated_df(spectral, timestamps):
    spectral_columns = [
        "NDVI_mean",
        "NDWI_mean",
        "NDRE_mean",
    ]

    df = spectral.merge(timestamps, on="Timestamp", how="right")
    interpolated_df = df[["Timestamp"] + spectral_columns].copy()
    interpolated_df[spectral_columns] = interpolated_df[spectral_columns].interpolate(method="pchip", limit_area="inside").bfill()

    return interpolated_df.dropna()

We create a function because we will be applying the interpolation to the anomaly dataset and the "normal"/expected dataset separately. Why? To prevent data leakage and noisy estimates. More specifically:

- There is a large chunk of observations missing around the hard split induced for the drought years 2018-2019. If we interpolate the full dataset first, ~90 days worth of observations will have _interpolated_ data points. It goes without saying that interpolation will absolutely **not** be able to reliably "estimate" 90 days of missing vegetation indices.
- Interpolating the data points around the split will carry over data from the _drought_ period to the _normal_ period. Our model will then have been exposed already to the drought conditions during its training. This is something that should rather be avoided.

We can theoretically solve this issue by building a regression model and filling large chunks (say $n>5$) of empty observations with predictions. For right now, we prioritise the unsupervised model first and therefore drop the empty observations.

In [ ]:
norm_interpolated_df = get_interpolated_df(spectral=norm_s_df, timestamps=norm_eng_ns_df["Timestamp"])
norm_interpolated_df.head()

In [ ]:
anom_interpolated_df = get_interpolated_df(spectral=anom_s_df, timestamps=anom_eng_ns_df["Timestamp"])
anom_interpolated_df.head()

In [ ]:
orig_plot_df = utils.merge_with_master(grouped_spectral_df.drop("Ag_year", axis=1), non_spectral_df)

# stack interpolated dfs for plotting
interpolated_plot_df = pd.concat([norm_interpolated_df, anom_interpolated_df], ignore_index=True)
interpolated_plot_df = interpolated_plot_df.sort_values("Timestamp")

plt.figure(figsize=(12, 6))
plt.title("Original vs. Interpolated NDVI")
sns.lineplot(x=orig_plot_df["Timestamp"], y=orig_plot_df["NDVI_mean"], label="Original", c="black")
sns.lineplot(x=interpolated_plot_df["Timestamp"], y=interpolated_plot_df["NDVI_mean"], label="Interpolated", c="red", alpha=0.5)
sns.lineplot()
plt.xlabel("Timestamp")
plt.ylabel("NDVI")
plt.legend()
plt.tight_layout()
plt.show()

---

## Full Model, no lagged variables or rolling statistics

As a baseline, we fit a GMM on all our engineered and remaining non-spectral variables. The log likelihood of the training sample will give us a good indication of how well the model fit to the data. Note that, since GMM's assume _continuous_ variables, we do not explicitly make use of the `in_season` or `Growth-Stage` features. We will operate under the assumption that this information is baked into `cummulative_GDD`.

In [ ]:
eng_features = [
    # Timestamp for merging
    'Timestamp',

    # Date features for plotting
    "shifted_doy",
    "Ag_year",

    # engineered features
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    #'VW_PC1',  
    #'Shallow_mean',
    'ST_PC1',

    # Categoricals not included
    #'Growth_Stage',
    'in_season',
]

non_spectral_features = [
    'Timestamp',
    'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    'precipitation',
    'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    'soil_temperature_level_4'
]


In [ ]:
# Get normal data
X_part1 = norm_eng_ns_df[eng_features]
X_part2 = norm_ns_df[non_spectral_features]
X_non_spectral_norm = pd.merge(X_part2, X_part1, on="Timestamp")

# Merge with interpolated spectral dataframe
X_norm = norm_interpolated_df.merge(X_non_spectral_norm, on="Timestamp", how="left")
X_norm['doy_sin'] = np.sin(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm['doy_cos'] = np.cos(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm["Ag_year"] = X_norm["Ag_year"].astype('int32')
X_norm = X_norm[X_norm["Ag_year"] != 2017]
X_norm[X_norm["in_season"] == 1]
X_norm = X_norm.drop("in_season", axis=1)
X_norm.shape

In [ ]:
# Get anomaly data

X_part1 = anom_eng_ns_df[eng_features]
X_part2 = anom_ns_df[non_spectral_features]
X_ns_anom = pd.merge(X_part2, X_part1, on="Timestamp")

# Merge with interpolated spectral dataframe
X_anom = anom_interpolated_df.merge(X_ns_anom, on="Timestamp", how="left")
X_anom['doy_sin'] = np.sin(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom['doy_cos'] = np.cos(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom["Ag_year"] = X_anom["Ag_year"].astype('int32')
X_anom = X_anom[X_anom["Ag_year"] != 2017]
X_anom[X_anom["in_season"] == 1]
X_anom = X_anom.drop("in_season", axis=1)
X_anom.shape

### Training

In [ ]:
scaler = StandardScaler()

In [ ]:
X = X_norm.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos"], axis=1).copy()

train_mask = X_norm["Ag_year"] < 2022
test_mask = X_norm["Ag_year"] >= 2022

X_train_raw = X[train_mask]
X_test_raw = X[test_mask]

scaler.fit(X_train_raw)

X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_raw.index)

X_train = pd.concat([X_train, X_norm.loc[train_mask, ["doy_sin", "doy_cos"]]], axis=1)
X_test = pd.concat([X_test, X_norm.loc[test_mask, ["doy_sin", "doy_cos"]]], axis=1)

In [ ]:
Xa = X_anom.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos"], axis=1).copy()

X_anom_scaled = scaler.transform(Xa)
X_anom_df = pd.DataFrame(X_anom_scaled, columns=Xa.columns, index=X_anom.index)
X_anom_df = pd.concat([X_anom_df, X_anom[["doy_sin", "doy_cos"]]], axis=1)

In [ ]:
covariance_types = ['full']
components = range(1, 50)
bics = np.zeros(shape=(len(covariance_types), len(components)))

min_bic = np.inf
min_bic_k = 0
best_cov = None
best_model = None


for i, cov in enumerate(covariance_types):
    for j, k in enumerate(components):
        gmm = GaussianMixture(n_components=k, covariance_type=cov, random_state=42, n_init=10, reg_covar=1e-3)
        gmm.fit(X_train)
        bic = gmm.bic(X_train)
        bics[i, j] = bic
        if bic < min_bic:
            min_bic = bic
            min_bic_k = k
            best_cov = cov
            best_model = gmm

print(f"Minimum BIC: {min_bic} with {min_bic_k} components and '{best_cov}' Covariance Matrices.")

In [ ]:
plt.figure(figsize=(10, 6))
for i, cov in enumerate(covariance_types):
    plt.plot(components, bics[i], label=f'{cov} covariance', marker='o')

plt.xlabel("Number of Components (k)")
plt.ylabel("BIC Score")
plt.title("GMM Model Selection via BIC")
plt.axvline(x=min_bic_k, color='red', linestyle='--', label=f'Best k ({min_bic_k})')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
bic_test_score = best_model.score(X_test)
bic_anom_score = best_model.score(X_anom_df)

bic_test_scores = best_model.score_samples(X_test)
bic_anom_scores = best_model.score_samples(X_anom_df)


In [ ]:


print(f"{'':<35}{'Lowest BIC model':<25}")
print("-" * 65)

print(f"{'Test data log-likelihood':<35}{bic_test_score:.2f}")
print(f"{'Anomaly data log-likelihood':<35}{bic_anom_score:.2f}")

We also create a density plot to better showcase the separation abilities of the model:

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6), sharey=False)
ax.set_title(f"Lowest BIC model log-likelihood ($k={min_bic_k}$)")
sns.kdeplot(bic_test_scores, label="Normal (Test)", fill=True, color="blue", alpha=0.5, ax=ax)
sns.kdeplot(bic_anom_scores, label="Anomaly", fill=True, color="orange", alpha=0.5, ax=ax)
ax.set_ylabel("Density")
ax.set_xlabel("log-Likelihood")
ax.legend()


In [ ]:
test_results = X_norm[test_mask][["Ag_year", "shifted_doy"]].copy()
test_results["log_likelihood"] = bic_test_scores
anom_results = X_anom[["Ag_year", "shifted_doy"]].copy()
anom_results["log_likelihood"] = bic_anom_scores

fig, ax = plt.subplots(figsize=(16, 6))

for year, group in test_results.groupby("Ag_year"):
    group = group.sort_values("shifted_doy")
    sns.lineplot(
        data=group,
        x="shifted_doy",
        y="log_likelihood",
        ax=ax,
        label=f"{str(year)} (test)",
        linewidth=3,
        color="blue",
        alpha=0.6
    )

for year, group in anom_results.groupby("Ag_year"):
    group = group.sort_values("shifted_doy")
    sns.lineplot(
        data=group,
        x="shifted_doy",
        y="log_likelihood",
        ax=ax,
        linestyle="--",
        label=f"Anomaly {year}",
        alpha=0.8
    )


ax.set_title(f"Lowest BIC model log-likelihood ($k={min_bic_k}$)")
ax.set_xlabel("Day of Year")
ax.set_ylabel("Log-Likelihood")
plt.show()

---

## Rolling and Lagged variables

While our first model does have pretty surprising performance, it does operate under two assumptions violated by the data:
1. **Independence between observations**.
2. **No temporal behaviour of variables**: The baseline GMM does not in any way use _past_ environmental data for _current_ observations during its clustering. This is a bad strategy for time series agricultural data because, for example, rain will not have a meaningful effect on a crop until a few days _after_ it has occurred. Similarly, a single day's drought will not put a plant under stress, but rather an _accumulation_ of days of drought.

While we cannot address the independence assumption since that is a defining characteristic of a GMM, we can include rolled statistics and lagging variables to force the model to learn on temporal data.

In [ ]:
roll_lag_features = [
    "Timestamp",
    
    # Temperature
    #"temperature_2m_lag_7",
    #"temperature_2m_lag_30",
    "temperature_2m_7D_mean",
    #"temperature_2m_30D_mean",

    # Solar radiation
    #"surface_solar_radiation_downwards_sum_lag_7",
    #"surface_solar_radiation_downwards_sum_lag_30",
    "surface_solar_radiation_downwards_sum_7D_mean",
    #"surface_solar_radiation_downwards_sum_30D_sum",

    # Precipitation
    #"precipitation_lag_7",
    #"precipitation_lag_30",
    "precipitation_7D_sum",
    #"precipitation_30D_sum",

    # Root-zone soil moisture (linear comb of first 3 volumetric soil water layers)
    #"root_weighted_soil_moisture_lag_7",
    #"root_weighted_soil_moisture_lag_30",
    #"root_weighted_soil_moisture_7D_mean",
    #"root_weighted_soil_moisture_30D_mean",

    # Volumetric soil water layers 1 and 2 (shallow) (first PC)
    #"VW_PC1_lag_7",
    #"VW_PC1_lag_30",
    #"VW_PC1_7D_mean",
    #"VW_PC1_30D_mean",

    # Soil temperatre levels 1-3 (shallow) PCs
    #"ST_PC1_lag_7",
    #"ST_PC1_lag_30",
    #"ST_PC1_7D_mean",
    #"ST_PC1_30D_mean",

    # Soil temperatre levels 1-3 (shallow) mean 
    #"Shallow_mean_lag_7",
    #"Shallow_mean_lag_30",
    #"Shallow_mean_7D_mean",
    #"Shallow_mean_30D_mean",

    # Deep volumetric Soil Water Layers 3-4
    #"volumetric_soil_water_layer_3_7D_mean",
    "volumetric_soil_water_layer_3_30D_mean",
    #"volumetric_soil_water_layer_4_7D_mean",
    #"volumetric_soil_water_layer_4_30D_mean",
    
    #'volumetric_soil_water_layer_3_lag_7',
    #'volumetric_soil_water_layer_3_lag_30',
    #'volumetric_soil_water_layer_4_lag_7',
    #'volumetric_soil_water_layer_4_lag_30',

    # Deep soil temperature level 4
    #"soil_temperature_level_4_7D_mean",
    "soil_temperature_level_4_30D_mean",
    #'soil_temperature_level_4_lag_7',
    #'soil_temperature_level_4_lag_30'

]

eng_features = [
    # Date/time variables 
    'Timestamp',
    "shifted_doy",
    #"doy",
    "Ag_year",

    
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    #'VW_PC1', 
    #'Shallow_mean',
    'ST_PC1',

    # Categoricals not valid for GMM
    #'in_season', 
    #'Growth_Stage',
    
]

non_spectral_features = [
    'Timestamp',
    #'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    #'precipitation',
    #'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    #'soil_temperature_level_4'
]



In [ ]:
# Get normal data
X_part1 = norm_eng_ns_df[eng_features]
X_part2 = norm_roll_lag_df[roll_lag_features].dropna()
X_part3 = norm_ns_df[non_spectral_features]

X_eng_norm = pd.merge(X_part2, X_part1, on="Timestamp")
X_non_spectral_norm = pd.merge(X_part3, X_eng_norm, on="Timestamp")


# Merge with interpolated spectral dataframe and drop NaNs from start of dataset caused by lags/rolls
X_norm = norm_interpolated_df.merge(X_non_spectral_norm, on="Timestamp", how="left").dropna()
X_norm['doy_sin'] = np.sin(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm['doy_cos'] = np.cos(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm["Ag_year"] = X_norm["Ag_year"].astype('int32')
X_norm = X_norm[X_norm["Ag_year"] != 2017]


In [ ]:
# Get anomaly data

X_part1 = anom_eng_ns_df[eng_features]
X_part2 = anom_roll_lag_df[roll_lag_features]
X_part3 = anom_ns_df[non_spectral_features]

X_eng_anom = pd.merge(X_part2, X_part1, on="Timestamp")
X_ns_anom = pd.merge(X_part3, X_eng_anom, on="Timestamp")

X_anom = anom_interpolated_df.merge(X_ns_anom, on="Timestamp", how="left").dropna()
X_anom['doy_sin'] = np.sin(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom['doy_cos'] = np.cos(2 * np.pi * X_anom['shifted_doy'] / 365.25)

### Curse of Dimensionality and High Correlation

It does not take much to see that fitting the model purely on all these features will be catastrophic. Many of them explain the exact same data in higher or lower dimensional representations. GMMs rely on distances and covariances across the feature space, making the models sensitive to multicollinearity and high dimensions (relative to the observations).

Our first step is to find the feature pairs that are highly correlated and figure out if we want to (or can) remove them from our feature space.

In [ ]:
corr = X_part2.corr().abs()
select_upper = np.triu(np.ones(corr.shape), k=1).astype(bool)
high_corr = corr.where(select_upper).stack().reset_index()
high_corr.columns = ["feature_1", "feature_2", "correlation"]
high_corr = high_corr[high_corr["correlation"] > 0.7].sort_values("correlation", ascending=False)
high_corr.nlargest(20, "correlation")

Even the first 20 highest correlated features have $\rho>0.95$. This will be highly problematic for the GMM so we need to start pruning. Our first step is to refer back to our feature engineering phase:
1. `ST_PC1` was made to be used _separately_ from `Shallow_mean`, since both explain the same data (just in a different representation).
2. Most 1-day lags and rolls can be removed as it is unlikely to provide much information to the model regarding the temporal behaviour of the crops.
3. The deeper layers of `soil_temperature_level` and `volumetric_soil_water_layer` hardly change as time goes on.
4. Rolling statistics typically hold structural states better than lags.

### Training

We now train our model on the lagged/rolled variables. This process is almost identical to that of our baseline.

In [ ]:
X = X_norm.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos"], axis=1).copy()

train_mask = X_norm["Ag_year"] < 2022
test_mask = X_norm["Ag_year"] >= 2022

X_train_raw = X[train_mask]
X_test_raw = X[test_mask]

scaler.fit(X_train_raw)

X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_raw.index)

X_train = pd.concat([X_train, X_norm.loc[train_mask, ["doy_sin", "doy_cos"]]], axis=1)
X_test = pd.concat([X_test, X_norm.loc[test_mask, ["doy_sin", "doy_cos"]]], axis=1)

In [ ]:
Xa = X_anom.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos"], axis=1).copy()

X_anom_scaled = scaler.transform(Xa)
X_anom_df = pd.DataFrame(X_anom_scaled, columns=Xa.columns, index=X_anom.index)
X_anom_df = pd.concat([X_anom_df, X_anom[["doy_sin", "doy_cos"]]], axis=1)


In [ ]:
X_train.columns

In [ ]:

covariance_types = ['full']
components = range(1, 50)
bics = np.zeros(shape=(len(covariance_types), len(components)))

min_bic = np.inf
min_bic_k = 0
best_cov = None
best_model = None


for i, cov in enumerate(covariance_types):
    for j, k in enumerate(components):
        gmm = GaussianMixture(n_components=k, covariance_type=cov, random_state=42, n_init=10, reg_covar=1e-4)
        gmm.fit(X_train)
        bic = gmm.bic(X_train)
        bics[i, j] = bic
        if bic < min_bic:
            min_bic = bic
            min_bic_k = k
            best_cov = cov
            best_model = gmm

print(f"Minimum BIC: {min_bic} with {min_bic_k} components and '{best_cov}' Covariance Matrices.")

In [ ]:
plt.figure(figsize=(10, 6))
for i, cov in enumerate(covariance_types):
    plt.plot(components, bics[i], label=f'{cov} covariance', marker='o')

plt.xlabel("Number of Components (k)")
plt.ylabel("BIC Score")
plt.title("GMM Model Selection via BIC")
plt.axvline(x=min_bic_k, color='red', linestyle='--', label=f'Best k ({min_bic_k})')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
elbow_k = 12
elbow_model = GaussianMixture(n_components=elbow_k, covariance_type=cov, random_state=42, n_init=6, reg_covar=1e-3).fit(X_train)

In [ ]:
bic_test_score = best_model.score(X_test)
bic_anom_score = best_model.score(X_anom_df)

elbow_test_score = elbow_model.score(X_test)
elbow_anom_score = elbow_model.score(X_anom_df)

print(f"{'Metric':<35}{'Lowest BIC model':<25}{'Elbow model':<25}")
print("-" * 85)

print(f"{'Test data log-likelihood':<35}{bic_test_score:<25.2f}{elbow_test_score:<25.2f}")
print(f"{'Anomaly data log-likelihood':<35}{bic_anom_score:<25.2f}{elbow_anom_score:<25.2f}")

In [ ]:
bic_test_scores = best_model.score_samples(X_test)
bic_anom_scores = best_model.score_samples(X_anom_df)

elbow_test_scores = elbow_model.score_samples(X_test)
elbow_anom_scores = elbow_model.score_samples(X_anom_df)

In [ ]:
# Create dataframes of sample scores for both models
bic_test_results = X_norm[test_mask][["Ag_year", "shifted_doy"]].copy()
bic_test_results["log_likelihood"] = bic_test_scores

bic_anom_results = X_anom[["Ag_year", "shifted_doy"]].copy()
bic_anom_results["log_likelihood"] = bic_anom_scores

elbow_test_results = X_norm[test_mask][["Ag_year", "shifted_doy"]].copy()
elbow_test_results["log_likelihood"] = elbow_test_scores

elbow_anom_results = X_anom[["Ag_year", "shifted_doy"]].copy()
elbow_anom_results["log_likelihood"] = elbow_anom_scores

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(20, 6))

# Plot one line per year
for year, group in bic_test_results.groupby("Ag_year"):
    group = group.sort_values("shifted_doy")
    sns.lineplot(
        data=group,
        x="shifted_doy",
        y="log_likelihood",
        ax=ax[0],
        label=f"{str(year)} (test)",
        linewidth=3,
        color="blue",
        alpha=0.6
    )

for year, group in bic_anom_results.groupby("Ag_year"):
    group = group.sort_values("shifted_doy")
    sns.lineplot(
        data=group,
        x="shifted_doy",
        y="log_likelihood",
        ax=ax[0],
        linestyle="--",
        label=f"Anomaly {year}",
        alpha=0.8
    )


ax[0].set_title(f"Lowest BIC model log-likelihood ($k={min_bic_k}$)")
ax[0].set_xlabel("Day of Year")
ax[0].set_ylabel("Log-Likelihood")

for year, group in elbow_test_results.groupby("Ag_year"):
    group = group.sort_values("shifted_doy")
    sns.lineplot(
        data=group,
        x="shifted_doy",
        y="log_likelihood",
        ax=ax[1],
        label=f"{str(year)} (test)",
        linewidth=3,
        color="blue",
        alpha=0.6
    )

for year, group in elbow_anom_results.groupby("Ag_year"):
    group = group.sort_values("shifted_doy")
    sns.lineplot(
        data=group,
        x="shifted_doy",
        y="log_likelihood",
        ax=ax[1],
        linestyle="--",
        label=f"Anomaly {year}",
        alpha=0.8
    )


ax[1].set_title(f"Lowest BIC model log-likelihood ($k={min_bic_k}$)")
ax[1].set_xlabel("Day of Year")
ax[1].set_ylabel("Log-Likelihood")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(20, 6))

sns.kdeplot(bic_test_scores, ax=ax[0], fill=True)
sns.kdeplot(bic_anom_scores, ax=ax[0], fill=True)
ax[0].set_title(f"Density of log-likelihood of lowest BIC model (k={min_bic_k})")
ax[0].set_xlabel("log-Likelihood")
ax[0].set_ylabel("Density")

sns.kdeplot(elbow_test_scores, ax=ax[1], fill=True)
sns.kdeplot(elbow_anom_scores, ax=ax[1], fill=True)
ax[1].set_title(f"Density of log-likelihood of elbow model (k={elbow_k})")
ax[1].set_xlabel("log-Likelihood")
ax[1].set_ylabel("Density")

plt.tight_layout()
plt.show()